In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day03" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w7" / "day03"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


In [3]:
from sqlalchemy import select
from app.core.config import get_settings
from app.db.session import session_scope
from app.integrations.factory import get_embedder
from app.rag import store
from app.rag.retriever import expand_queries
settings = get_settings()
QUESTION = '1~2급 직원의 숙박비 상한은 얼마인가요?'

def candidates_for(question: str) -> list[dict]:
    embedder = get_embedder()
    with session_scope() as session:
        seen: dict[int, dict] = {}
        for one in expand_queries(question, settings.multi_query_n):
            for row in store.vector_search(session, embedder.embed_query(one), limit=settings.retrieval_top_k):
                seen.setdefault(row['chunk_id'], row)
        vector_rows = sorted(seen.values(), key=lambda r: r['score'], reverse=True)
        keyword_rows = store.keyword_search(session, question, limit=settings.retrieval_top_k)
    return store.fuse(vector_rows, keyword_rows)[:settings.retrieval_top_k]
rows = candidates_for(QUESTION)
for (n, row) in enumerate(rows, 1):
    if n == settings.rerank_top_n + 1:
        print(f"{'─' * 27} 여기서 잘립니다 (rerank_top_n = {settings.rerank_top_n})")
    print(f"{n:>2}  {row['doc_id']}  {row['version']}  {row['kind']:<2}  {row['locator'][:22]:<22} {row['score']:.3f}")

12:05:40 INFO     sentence_transformers.base.model: No device provided, using cpu
12:05:40 INFO     sentence_transformers.base.model: Loading SentenceTransformer model from C:\hw\hw\python_basic\models\bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

12:05:46 INFO     app.integrations.local_embed: 로컬 임베딩 모델 준비: C:\hw\hw\python_basic\models\bge-m3 / 1024차원


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 1  DOC-HR-014  v2.0  표   표2                     0.714
 2  DOC-HR-014  v1.1  표   표2                     0.690
 3  DOC-HR-014  v1.1  조항  제3장 제11조(숙박비) · 별표1    0.648
─────────────────────────── 여기서 잘립니다 (rerank_top_n = 3)
 4  DOC-HR-014  v2.0  조항  제3장 제2절 제14조(숙박비) · 별표 0.691
 5  DOC-HR-014  v2.0  표   표3                     0.647
 6  DOC-HR-014  v2.0  조항  제3장 제1절 제11조(일비) · 별표1 0.601
 7  DOC-HR-014  v1.1  조항  제3장 제8조(일비) · 별표1      0.594
 8  DOC-HR-014  v2.0  조항  제3장 제2절 제15조(지역 구분) ·  0.590
 9  DOC-HR-014  v2.0  조항  제3장 제2절 제16조(숙박비의 예외)  0.659
10  DOC-HR-014  v2.0  조항  제3장 제1절 제12조(식비) · 별표1 0.653
11  DOC-HR-014  v1.1  조항  제1장 제4조(여비의 종류) · 별표1  0.000
12  DOC-HR-014  v1.1  조항  제3장 제9조(식비)            0.643


In [4]:
from app.rag.reranker import rerank, score as lexical_score

before = [(r['doc_id'], r['locator'][:12], r['score'], lexical_score(QUESTION, r['text']), 0.03 if r['kind'] == '표' else 0.0) for r in rows]

after = rerank(QUESTION, rows, top_n=len(rows))

print('  [리랭킹 전]                                  [리랭킹 후]')
for (n, (left, right)) in enumerate(zip(before, after), 1):
    (doc_id, locator, vec, lex, bonus) = left
    print(f"{n:>2} {doc_id} {locator:<12} 벡터 {vec:.3f}   {n:>2} {right['doc_id']} {right['locator'][:12]:<12} rerank {right['rerank_score']:.3f}  (벡터 {right['score']:.3f})")


12:05:55 INFO     sentence_transformers.base.model: No device provided, using cpu
12:05:55 INFO     sentence_transformers.base.model: No modules.json found for ..\models\bge-reranker-v2-m3, initializing a new CrossEncoder model.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

12:05:58 INFO     app.integrations.local_rerank: 로컬 리랭커 준비 : ..\models\bge-reranker-v2-m3, 최대 512토큰
  [리랭킹 전]                                  [리랭킹 후]
 1 DOC-HR-014 표2           벡터 0.714    1 DOC-HR-014 표2           rerank 0.893  (벡터 0.690)
 2 DOC-HR-014 표2           벡터 0.690    2 DOC-HR-014 표2           rerank 0.868  (벡터 0.714)
 3 DOC-HR-014 제3장 제11조(숙박비 벡터 0.648    3 DOC-HR-014 제3장 제2절 제14조 rerank 0.516  (벡터 0.691)
 4 DOC-HR-014 제3장 제2절 제14조 벡터 0.691    4 DOC-HR-014 표3           rerank 0.494  (벡터 0.647)
 5 DOC-HR-014 표3           벡터 0.647    5 DOC-HR-014 제3장 제2절 제16조 rerank 0.240  (벡터 0.659)
 6 DOC-HR-014 제3장 제1절 제11조 벡터 0.601    6 DOC-HR-014 제3장 제11조(숙박비 rerank 0.150  (벡터 0.648)
 7 DOC-HR-014 제3장 제8조(일비)  벡터 0.594    7 DOC-HR-014 제3장 제1절 제11조 rerank 0.009  (벡터 0.601)
 8 DOC-HR-014 제3장 제2절 제15조 벡터 0.590    8 DOC-HR-014 제3장 제8조(일비)  rerank 0.008  (벡터 0.594)
 9 DOC-HR-014 제3장 제2절 제16조 벡터 0.659    9 DOC-HR-014 제3장 제1절 제12조 rerank 0.003  (벡터 0.653)
10 DOC-HR-014 제3장 제1절 제12조 벡터 0.653   1

In [5]:
import importlib
import time
from app.core import config
from app.integrations import factory, ports
from app.rag import reranker

for module in (config, ports, factory, reranker):
    importlib.reload(module)
    
config.get_settings.cache_clear()
print('RERANKER :', config.get_settings().reranker)

start = time.perf_counter()
model = factory.get_cross_encoder()
print(f'모델을 열었습니다 ({time.perf_counter() - start:.1f}s)')

QUESTION = '해외 출장 시 비즈니스석 탑승 기준이 어떻게 되나요?'
PIECES = ['제7조(출장 신청) ① 출장을 하려는 임직원은 출장 개시일 3일 전까지 출장 신청서를 제출한다.', '제14조(숙박비) ② 광역시 지역의 숙박비는 1박당 70,000원을 상한으로 한다.']
start = time.perf_counter()
scores = model.score_pairs(QUESTION, PIECES)

for piece, s in zip(PIECES, scores):
    print(f'{s:.4f}  {piece[:24]}')
print(f'두 쌍 {time.perf_counter() - start:.2f}s · 합격선 {config.get_settings().rerank_threshold}')


RERANKER : cross_encoder
12:06:08 INFO     sentence_transformers.base.model: No device provided, using cpu
12:06:08 INFO     sentence_transformers.base.model: No modules.json found for ..\models\bge-reranker-v2-m3, initializing a new CrossEncoder model.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

12:06:13 INFO     app.integrations.local_rerank: 로컬 리랭커 준비 : ..\models\bge-reranker-v2-m3, 최대 512토큰
모델을 열었습니다 (5.5s)
0.0004  제7조(출장 신청) ① 출장을 하려는 임직원
0.0000  제14조(숙박비) ② 광역시 지역의 숙박비는
두 쌍 0.40s · 합격선 0.2


In [6]:
# 노트북 
from app.rag import policy as pol
VIEWER = {'name': '김민준', 'dept_id': 'INFRA2', 'clearance': '일반'}
for label, question in (('[기본]', QUESTION), ('[「개정 전」]', f'개정 전 {QUESTION}')):
    found = candidates_for(question)
    kept, blocked = pol.filter_candidates(found, user=VIEWER, question=question)
    print(f'{label:<12} 후보 {len(found)}건 → 통과 {len(kept)}건')
    for one in list(dict.fromkeys(blocked))[:4]:
        print(f'   거른 사유 : {one}')


14:19:20 INFO     sentence_transformers.base.model: No device provided, using cpu
14:19:20 INFO     sentence_transformers.base.model: Loading SentenceTransformer model from C:\hw\hw\python_basic\models\bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

14:19:22 INFO     app.integrations.local_embed: 로컬 임베딩 모델 준비: C:\hw\hw\python_basic\models\bge-m3 / 1024차원


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

[기본]         후보 12건 → 통과 7건
   거른 사유 : 국내출장 여비 규정 v1.1 - 만료 버전


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[「개정 전」]     후보 12건 → 통과 12건
